# Notebook 05 — Food Price Forecasting

**Objective:** Forecast Kenya food price index (CPI) using KNBS monthly data.

**Models:** ARIMA baseline → Facebook Prophet → LSTM (stretch goal)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from src.models import evaluate_regression, save_model
from src.utils import section

knbs = pd.read_csv("data/processed/knbs_cpi_structured.csv")
print("KNBS structured shape:", knbs.shape)
knbs.head()

## Step 1 — Stationarity Check

In [ ]:
section("STATIONARITY — ADF Test")
if "overall_cpi" in knbs.columns:
    cpi_series = knbs.dropna(subset=["overall_cpi"]).set_index("date")["overall_cpi"]
    cpi_series.index = pd.to_datetime(cpi_series.index)
    
    result = adfuller(cpi_series.dropna())
    print(f"ADF Statistic : {result[0]:.4f}")
    print(f"p-value       : {result[1]:.4f}")
    print(f"Is stationary : {result[1] < 0.05}")
else:
    print("overall_cpi column not found — check KNBS cleaning in notebook 02")

## Step 2 — ARIMA Baseline

In [ ]:
section("BASELINE — ARIMA(1,1,1)")
if "overall_cpi" in knbs.columns:
    cpi = knbs.dropna(subset=["overall_cpi"]).sort_values("date")
    cpi["date"] = pd.to_datetime(cpi["date"])
    cpi = cpi.set_index("date")["overall_cpi"]

    # Train/test split — last 6 months as test
    train, test = cpi.iloc[:-6], cpi.iloc[-6:]

    model_arima = ARIMA(train, order=(1, 1, 1))
    fitted_arima = model_arima.fit()
    forecast_arima = fitted_arima.forecast(steps=len(test))

    arima_results = evaluate_regression(test, forecast_arima, "ARIMA(1,1,1)")

    plt.figure(figsize=(12, 5))
    plt.plot(train.index, train, label="Training data", color="#1565C0")
    plt.plot(test.index, test, label="Actual", color="#2E7D32", linewidth=2)
    plt.plot(test.index, forecast_arima, label="ARIMA Forecast",
             color="#E65100", linestyle="--", linewidth=2)
    plt.title("KNBS CPI — ARIMA Baseline Forecast")
    plt.xlabel("Date")
    plt.ylabel("CPI Index")
    plt.legend()
    plt.tight_layout()
    plt.savefig("reports/figures/08_arima_forecast.png", dpi=150, bbox_inches="tight")
    plt.show()

## Step 3 — Prophet Model

In [ ]:
section("MODEL 2 — Facebook Prophet")
try:
    from prophet import Prophet

    if "overall_cpi" in knbs.columns:
        # Prophet requires columns: ds (date) and y (value)
        prophet_df = knbs.dropna(subset=["overall_cpi"]).copy()
        prophet_df["ds"] = pd.to_datetime(prophet_df["date"])
        prophet_df["y"]  = prophet_df["overall_cpi"]
        
        train_p = prophet_df.iloc[:-6]
        test_p  = prophet_df.iloc[-6:]

        m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                    daily_seasonality=False, changepoint_prior_scale=0.1)
        m.fit(train_p[["ds", "y"]])

        # Forecast
        future   = m.make_future_dataframe(periods=6, freq="MS")
        forecast  = m.predict(future)
        
        forecast_vals = forecast.iloc[-6:]["yhat"].values
        prophet_results = evaluate_regression(test_p["y"].values, forecast_vals, "Prophet")

        fig = m.plot(forecast)
        plt.title("Kenya CPI — Prophet Forecast with Uncertainty Intervals")
        plt.tight_layout()
        plt.savefig("reports/figures/09_prophet_forecast.png", dpi=150, bbox_inches="tight")
        plt.show()

except ImportError:
    print("Prophet not installed — run: pip install prophet")